In [ ]:
import os

import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform

from psm_final import BetaVAEAnalysis, Algonauts, TripleN, shared_stimuli

ALGONAUTS_DIR = os.environ["ALGONAUTS_DIR"]
TRIPLE_N_DIR = os.environ["TRIPLE_N_DIR"]
NSD_MAT = "../nsd_expdesign.mat"

In [ ]:
shared_ids = shared_stimuli(NSD_MAT)

In [ ]:
algonauts = Algonauts(ALGONAUTS_DIR, shared_ids)
triple_n = TripleN(TRIPLE_N_DIR)
bvae_analysis = BetaVAEAnalysis(triple_n_path="/media/chuddy/Extreme SSD/data/triple-N/", model_path="../results/beta_vae/latent_512_beta_2.0_epochs_50_seed_42/vae.pth")
bvae_rdm = bvae_analysis.rdm()

In [ ]:
plt.figure(figsize=(5,4))
plt.matshow(squareform(bvae_rdm))
plt.title(f'RDM for Beta-VAE')
plt.colorbar()
plt.show()

In [ ]:
from scipy.stats import spearmanr

tripleN_rdm = triple_n.compute_rdm(area_label="V4")

rho, pval = spearmanr(bvae_rdm, tripleN_rdm)
print(f"Spearman correlation between Beta-VAE RDM and Triple-N RDM: rho = {rho:.4f}, p-value = {pval:.4e}")

In [ ]:
import numpy as np

subjects = range(1, 9)
matched = [set(algonauts.shared_stimuli_indices(s, shared_ids)[0]) for s in subjects]
common = sorted(set.intersection(*matched))
stim_idx = np.array(TripleN.nsd_to_stim_index(common))-1

bvae_rdm_common = bvae_analysis.rdm(indices=stim_idx)
algonauts_rdm_common = algonauts.compute_rdm(subject=7, indices=common)

rho_common, pval_common = spearmanr(bvae_rdm_common, algonauts_rdm_common)
print(f"Spearman correlation between Beta-VAE RDM and Algonauts RDM (common stimuli): rho = {rho_common:.4f}, p-value = {pval_common:.4e}")

# Model x brain RDM correlation tables

Spearman correlation of every trained Beta-VAE model's RDM against four groupings of brain data: Algonauts fMRI **ROIs**, Triple-N **area labels**, Triple-N **individual macaques** (all units pooled), and Triple-N unit **firing-preference types** (Face/Body/Object-tuned units). Every RDM is computed over the **same shared stimuli in one shared order**, so they all line up. Same construction as the ROI x area table in `data_exploration.ipynb`, with the models on the rows.

In [ ]:
# RDM similarity tables: every Beta-VAE model (rows) vs. several groupings of
# brain data (columns):
#   1. Algonauts fMRI ROIs
#   2. Triple-N area labels
#   3. Triple-N individual macaques (whole population, all units pooled)
#   4. Triple-N unit firing-preference types (Face / Body / Object tuned units)
# All RDMs are computed over the SAME shared stimuli in one shared order so
# every pair lines up for Spearman comparison.
import glob
import re

import numpy as np
from scipy.stats import spearmanr

from psm_final import BetaVAEAnalysis, noise_ceiling

SUBJECTS = range(1, 9)

# --- shared stimuli present in EVERY subject's split AND with a Triple-N mapping ---
matched = [set(algonauts.shared_stimuli_indices(s, shared_ids)[0]) for s in SUBJECTS]
common = sorted(set.intersection(*matched))

stim_idx = TripleN.nsd_to_stim_index(common)          # 1-based stim_index, None if unmapped
keep = [k for k, s in enumerate(stim_idx) if s is not None]
common_aligned = [common[k] for k in keep]            # NSD ids     -> Algonauts indexing
stim_aligned = [stim_idx[k] for k in keep]            # 1-based stim_index -> Triple-N indexing
model_indices = np.array(stim_aligned) - 1            # 0-based positions into StimuliNNN
print(f"{len(common_aligned)} shared stimuli used for every RDM")

# --- Beta-VAE model RDMs: one per checkpoint, over the shared stimuli/order ---
def beta_of(path):
    return float(re.search(r"beta_([0-9.]+)_epochs", path).group(1))

model_paths = sorted(glob.glob("../results/beta_vae/*/vae.pth"), key=beta_of)
model_rdms, model_labels = {}, []
for path in model_paths:
    label = f"β={beta_of(path):g}"
    analysis = BetaVAEAnalysis(triple_n_path=TRIPLE_N_DIR, model_path=path)
    model_rdms[label] = analysis.rdm(indices=model_indices)
    model_labels.append(label)

# --- Algonauts: per-subject ROI RDMs -> group-mean RDM + noise ceiling ---
# NOTE: compute_rdm reloads the subject fMRI on each call, so this re-reads per (subject, ROI).
algo_rdms, algo_nc = {}, {}
for roi in Algonauts.ALGO_ROIS:
    per_subj = [algonauts.compute_rdm(subject=s, indices=common_aligned, roi=roi) for s in SUBJECTS]
    per_subj = [r for r in per_subj if r.std() > 0]   # drop subjects where the ROI is empty
    if len(per_subj) < 2:                             # need >=2 subjects for a noise ceiling
        continue
    per_subj = np.vstack(per_subj)
    algo_rdms[roi] = per_subj.mean(axis=0)
    algo_nc[roi] = noise_ceiling(per_subj)            # (lower, upper)

# --- Triple-N: per-area-label RDM (units pooled) + noise ceiling (across macaques) ---
macaques = sorted(triple_n.units["macaque"].unique())
area_labels = sorted(triple_n.units["area_label"].unique())
triple_rdms, triple_nc = {}, {}
for label in area_labels:
    try:
        triple_rdms[label] = triple_n.compute_rdm(area_label=label, indices=stim_aligned)
    except ValueError:
        continue                                      # fewer than 2 units for this label
    per_macaque = []
    for m in macaques:
        try:
            r = triple_n.compute_rdm(area_label=label, macaque=m, indices=stim_aligned)
        except ValueError:
            continue                                  # this macaque has too few units here
        if r.std() > 0:
            per_macaque.append(r)
    if len(per_macaque) >= 2:                          # need >=2 macaques for a noise ceiling
        triple_nc[label] = noise_ceiling(np.vstack(per_macaque))

# --- Triple-N: whole-population RDM per individual macaque (all units pooled) +
#     within-macaque noise ceiling (reliability across that macaque's sessions) ---
macaque_rdms, macaque_nc = {}, {}
for m in macaques:
    try:
        macaque_rdms[m] = triple_n.compute_rdm(macaque=m, indices=stim_aligned)
    except ValueError:
        continue                                      # fewer than 2 units for this macaque
    sessions = sorted(triple_n.units.loc[triple_n.units["macaque"] == m, "session"].unique())
    per_session = []
    for s in sessions:
        try:
            r = triple_n.compute_rdm(macaque=m, session=s, indices=stim_aligned)
        except ValueError:
            continue                                  # this session has too few units
        if r.std() > 0:
            per_session.append(r)
    if len(per_session) >= 2:                          # need >=2 sessions for a noise ceiling
        macaque_nc[m] = noise_ceiling(np.vstack(per_session))

# --- Triple-N: RDM per unit firing-preference type (F/B/O), pooled across all
#     macaques, + noise ceiling across macaques ---
PREF_NAMES = {"F": "Face-pref", "B": "Body-pref", "O": "Object-pref"}
preferences = sorted(triple_n.units["preference"].unique())
pref_cols = [PREF_NAMES.get(p, p) for p in preferences]
pref_rdms, pref_nc = {}, {}
for p in preferences:
    name = PREF_NAMES.get(p, p)
    try:
        pref_rdms[name] = triple_n.compute_rdm(preference=p, indices=stim_aligned)
    except ValueError:
        continue                                      # fewer than 2 units with this preference
    per_macaque = []
    for m in macaques:
        try:
            r = triple_n.compute_rdm(preference=p, macaque=m, indices=stim_aligned)
        except ValueError:
            continue
        if r.std() > 0:
            per_macaque.append(r)
    if len(per_macaque) >= 2:
        pref_nc[name] = noise_ceiling(np.vstack(per_macaque))


def plot_model_corr_table(brain_rdms, brain_cols, brain_nc, xlabel, title):
    """Annotated heatmap: Beta-VAE models (rows) x brain regions (cols) Spearman
    correlation. A noise-ceiling row (per-column upper bound) is appended when
    `brain_nc` is non-empty."""
    cols = [c for c in brain_cols if c in brain_rdms]
    corr = np.array([[spearmanr(model_rdms[m], brain_rdms[c])[0] for c in cols]
                     for m in model_labels])

    show_ceiling = bool(brain_nc)
    M = np.full((len(model_labels) + int(show_ceiling), len(cols)), np.nan)
    M[:len(model_labels), :] = corr
    row_labels = list(model_labels)
    if show_ceiling:
        M[-1, :] = [brain_nc.get(c, (np.nan, np.nan))[1] for c in cols]   # ceiling upper bound -> row
        row_labels = row_labels + ["noise ceiling"]

    vmax = np.nanmax(np.abs(M))
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("lightgray")                          # NaN cells (missing ceiling) shown gray
    fig, ax = plt.subplots(figsize=(0.62 * len(cols) + 3, 0.5 * len(row_labels) + 2))
    im = ax.imshow(M, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)
    if show_ceiling:
        ax.axhline(len(model_labels) - 0.5, color="k", lw=1.5)   # separate the noise-ceiling row
    ax.set_xlabel(xlabel); ax.set_ylabel("Beta-VAE model")
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(v) > 0.6 * vmax else "black")
    fig.colorbar(im, ax=ax, label="Spearman rho")
    plt.tight_layout()
    plt.show()
    return fig


fig_algo = plot_model_corr_table(
    algo_rdms, Algonauts.ALGO_ROIS, algo_nc,
    xlabel="Algonauts fMRI ROI",
    title="RDM Spearman correlation:  Beta-VAE model  x  Algonauts ROI  (+ noise ceiling)",
)
fig_area = plot_model_corr_table(
    triple_rdms, area_labels, triple_nc,
    xlabel="Triple-N area label",
    title="RDM Spearman correlation:  Beta-VAE model  x  Triple-N area  (+ noise ceiling)",
)
fig_macaque = plot_model_corr_table(
    macaque_rdms, macaques, macaque_nc,
    xlabel="Triple-N macaque (all units)",
    title="RDM Spearman correlation:  Beta-VAE model  x  Triple-N macaque  (+ across-session noise ceiling)",
)
fig_pref = plot_model_corr_table(
    pref_rdms, pref_cols, pref_nc,
    xlabel="Triple-N unit firing preference",
    title="RDM Spearman correlation:  Beta-VAE model  x  firing-preference type  (+ noise ceiling)",
)

In [ ]:
fig_algo.savefig(
    "../figures/bvae_algo_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_area.savefig(
    "../figures/bvae_tn_area_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_macaque.savefig(
    "../figures/bvae_tn_macaque_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_pref.savefig(
    "../figures/bvae_tn_pref_heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)